# Project 3 - Single-Slide Exploration To Weak Supervision

This notebook uses the top mixed stack for a beginner-to-intermediate pathology workflow:

1. `LazySlide`
2. `Slideflow`
3. `CTransPath`-style feature extraction plus `Attention MIL` / `CLAM` planning


## VM Access

Windows Command Prompt:

```cmd
ssh -i "%USERPROFILE%\.ssh\evolet_rsa" pardeep@34.59.145.240
```

Linux setup after login:

```bash
source /opt/miniforge3/etc/profile.d/conda.sh
conda activate /opt/miniforge3/envs/pathology310
```


In [ ]:
from pathlib import Path
import json

SINGLE_SLIDE = Path("/path/to/open_slide.svs")
COHORT_ANNOTATIONS = Path("../project_1_slideflow_msi_tcga_crc/annotations/tcga_crc_msi_annotations.csv")
ENRICHED_ANNOTATIONS = Path("../project_1_slideflow_msi_tcga_crc/annotations/tcga_crc_msi_annotations_enriched_cbioportal_pub.csv")
OUTPUT_DIR = Path("./outputs/project_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
stack_report = {}

try:
    import lazyslide as zs
    stack_report["lazyslide"] = getattr(zs, "__version__", "installed")
except Exception as exc:
    stack_report["lazyslide_error"] = str(exc)

try:
    import slideflow as sf
    stack_report["slideflow"] = getattr(sf, "__version__", "installed")
except Exception as exc:
    stack_report["slideflow_error"] = str(exc)

try:
    import timm
    stack_report["timm"] = getattr(timm, "__version__", "installed")
except Exception as exc:
    stack_report["timm_error"] = str(exc)

stack_report


## Top 3 Mixed Design

- `LazySlide`: single-slide inspection and region understanding
- `Slideflow`: patch extraction, manifests, dataset movement, training workflow
- `CTransPath`-style features + `Attention MIL` / `CLAM`: weak supervision stage


In [ ]:
project_design = {
    "single_slide_phase": {
        "library": "LazySlide",
        "goal": [
            "understand tissue distribution",
            "inspect morphology-rich regions",
            "decide what should become training signal"
        ]
    },
    "cohort_phase": {
        "library": "Slideflow",
        "goal": [
            "prepare manifests",
            "extract patches or bags",
            "connect labels to slides"
        ]
    },
    "weak_supervision_phase": {
        "feature_backbone": "CTransPath-style",
        "algorithms": ["Attention MIL", "CLAM"],
        "goal": [
            "learn from slide-level labels",
            "compare bag-level behavior against original slide intuition"
        ]
    }
}

(OUTPUT_DIR / "project_design.json").write_text(json.dumps(project_design, indent=2), encoding="utf-8")
project_design


In [ ]:
pseudo_pipeline = {
    "step_1": "Inspect one open SVS slide with LazySlide",
    "step_2": "Document ROI intuition and tissue-rich areas",
    "step_3": "Build a small cohort from curated annotations",
    "step_4": "Use Slideflow-compatible manifests and patch flow",
    "step_5": "Extract pathology features with a CTransPath-style encoder",
    "step_6": "Train Attention MIL or CLAM on slide-level labels",
    "step_7": "Review whether predicted focus areas match the original single-slide intuition"
}

pseudo_pipeline


## Strong Next Experiments

- compare generic ImageNet features vs pathology-specific `CTransPath`
- compare `Attention MIL` vs `CLAM`
- compare single-slide ROI intuition vs model attention output
- use TCGA MSI labels from the curated annotation tables already present in this repo
